# Practice 46 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — penguins

Load `sns.load_dataset('penguins')`. Drop nulls.

- Extract `species_code`: the first letter of `species` (e.g. `'Adelie'` → `'A'`).
- Convert `species_code` to `category`.
- `groupby(['island', 'species_code'])` → mean `flipper_length_mm`. Unstack to get an island × species_code table.
- Which `(island, species_code)` combination has the longest mean flipper? Use `.stack()` on your result to get back a Series, then `.idxmax()`.

In [8]:
penguins = sns.load_dataset('penguins').dropna()

# Your code here

penguins['species_code'] = penguins['species'].str.extract(r'^(\w)')
penguins['species_code'] = penguins['species_code'].astype('category')
t = penguins.groupby(['island','species_code'], observed=True)['flipper_length_mm'].mean().unstack()
st = t.stack()
st.idxmax()


('Biscoe', 'G')

---
## Level 2 — student scores

Use the DataFrame below. No step-by-step — figure out the approach yourself.

- `full_name` is in `'Lastname, Firstname'` format. Split it into `last_name` and `first_name`.
- The three score columns share a naming pattern. Use a list comprehension to collect them.
- Stack the score columns into long format. The result should have a `(student, quarter)` index.
- What is the mean score per quarter? Who has the highest mean score across all quarters?

In [26]:
records = pd.DataFrame({
    'full_name': ['Smith, John', 'Doe, Jane', 'Brown, Bob', 'Lee, Alice', 'Kim, Chris'],
    'score_q1':  [88, 72, 95, 61, 83],
    'score_q2':  [91, 68, 88, 74, 79],
    'score_q3':  [85, 77, 92, 69, 88],
})

# Your code here

records['last_name'] = records['full_name'].str.extract(r'(\w+),')
records['first_name'] = records['full_name'].str.extract(r', (\w+)')

scs = [cn for cn in records.columns if 'score_q' in cn]

records[scs].set_index(records['full_name']).stack().rename_axis(['student','quarter'])

records[scs].mean(axis=0)

records[scs].set_index(records['full_name']).mean(axis = 1).idxmax()




'Brown, Bob'

---
## Level 3 — transaction pipeline

Each `txn_id` encodes year, a transaction number, and currency (e.g. `'TXN-2024-001-USD'`). `merchant` is formatted as `'Name-Region'`.

Write a `.pipe()` chain:
- **`parse(df)`** — extract `year`, `txn_num` (e.g. `'001'`), `currency`, `merchant_name` (before the `-`), `merchant_region` (after the `-`) from their respective columns.
- **`enrich(df)`** — add `amount_tier`: `'small'` (< 1000), `'mid'` (1000–2999), `'large'` (≥ 3000). Add `is_usd`: True where `currency` is `'USD'`.

After the chain:
- Mean `amount` per `currency`, sorted descending.
- Which `merchant_region` has the highest total `amount`?
- Among `'large'` transactions, what fraction are `is_usd`?
- `{currency: count}` — use a dict comprehension from a groupby.

In [37]:
transactions = pd.DataFrame({
    'txn_id':   ['TXN-2024-001-USD', 'TXN-2024-002-EUR', 'TXN-2023-003-USD',
                 'TXN-2024-004-GBP', 'TXN-2023-005-EUR', 'TXN-2024-006-USD',
                 'TXN-2023-007-GBP', 'TXN-2024-008-EUR'],
    'amount':   [1250.50, 890.00, 3400.75, 560.25, 2100.00, 4500.00, 1875.00, 3200.00],
    'merchant': ['Amazon-US', 'Booking-EU', 'Apple-US', 'ASOS-UK',
                 'Spotify-EU', 'Apple-US', 'ASOS-UK', 'Booking-EU'],
})

# Your code here


def parse(df):
    df['year'] = df['txn_id'].str.extract(r'TXN-(\d+)-')
    df['txn_num'] = df['txn_id'].str.extract(r'TXN-(?:\d+)-(\d{3})')
    df['currency'] = df['txn_id'].str.extract(r'-(\w+)$')
    df['merchant_name'] = df['merchant'].str.extract(r'^(\w+)-')
    df['merchant_region'] = df['merchant'].str.extract(r'-(\w+)$')
    return df
def enrich(df):
    df['amount_tier'] = pd.cut(df['amount'], bins=[0,1000,3000,np.inf], labels=['small','mid','large'],
                               right=False)
    df['is_usd'] = df['currency'] == 'USD'
    return df


res = transactions.copy().pipe(parse).pipe(enrich)

res.groupby('currency')['amount'].mean().sort_values(ascending = False)
res.groupby('merchant_region')['amount'].sum().idxmax()
res.loc[res['amount_tier']=='large','is_usd'].mean()
g = res.groupby('currency').size()
{c:count for c, count in zip(g.index, g)}

{'EUR': 3, 'GBP': 2, 'USD': 3}